# SunShift - Solar Generation Forecasting Model (Retrained)
**LECO Nawala Substation | Colombo, Sri Lanka | Jan-Feb 2026 (clean dataset)**

> This notebook **retrains** the SunShift solar-forecasting model on a freshly
> acquired, corruption-free LECO export covering **January and February 2026**
> (two monthly files). The prior training set contained corrupted records; the
> pipeline below is otherwise identical to the original.

### Pipeline Overview
1. Load & clean the two monthly LECO 15-min load-profile files (Jan + Feb 2026)
2. Identify real solar customers by daytime-only export signature
3. Aggregate substation-level solar export as the ground-truth target
4. Fetch matching 15-min weather data from the Open-Meteo historical forecast API
5. Engineer features (solar geometry, radiation lags, cyclic time encodings)
6. Train a **LightGBM** regression model (daytime-only, log1p target)
7. Export to **ONNX opset 12** for consumption by ML.NET in the mobile-app backend

> **Target variable:** per-customer average solar kW exported to grid
> **Primary weather features:** Shortwave GHI, DNI, DHI, cloud cover, temperature, sunshine duration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import warnings
import json
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (15, 5)

# Use the current directory as the data root (relative to notebook location)
DATA_DIR = Path('.')
(DATA_DIR / 'images').mkdir(exist_ok=True)   # ensure images/ subfolder exists

# Data period for this clean dataset: 1 Jan 2026 -> 28 Feb 2026
PERIOD_START  = '2026-01-01'
PERIOD_END    = '2026-02-28'
PERIOD_END_TS = '2026-02-28 23:45'

print("Data directory:", DATA_DIR.resolve())
print("Files:", [f.name for f in DATA_DIR.iterdir()])

## 1. Load & Explore LECO Data
The new dataset arrives as **two monthly files** (`LP-AZ0247-2026-01.csv`, `LP-AZ0247-2026-02.csv`). They are concatenated and de-duplicated on `(serial, datetime)`.

In [ ]:
# Locate and load both monthly LECO CSVs
csv_paths = sorted(DATA_DIR.glob('LP-AZ0247-2026-*.csv'))
print("Loading:", [p.name for p in csv_paths])

frames = [pd.read_csv(p, dtype={'SERIAL': str}, low_memory=False) for p in csv_paths]
raw = pd.concat(frames, ignore_index=True)

# --- Column rename map ---
col_map = {
    'SERIAL':                       'serial',
    'TIMESTAMP':                    'timestamp',
    'OBIS':                         'obis',
    'DATE':                         'date_str',
    'TIME':                         'time_str',
    'AVG._IMPORT_KW (kW)':          'avg_import_kw',
    'IMPORT_KWH (kWh)':             'import_kwh',
    'AVG._EXPORT_KW (kW)':          'avg_export_kw',
    'EXPORT_KWH (kWh)':             'export_kwh',
    'IMPORT_KVARH (kvarh)':         'import_kvarh',
    'EXPORT_KVARH (kvarh)':         'export_kvarh',
    'AVG._IMPORT_KVA (kVA)':        'avg_import_kva',
    'AVG._EXPORT_KVA (kVA)':        'avg_export_kva',
    'PHASE_A_INST._CURRENT (A)':    'ia',
    'PHASE_B_INST._CURRENT (A)':    'ib',
    'PHASE_C_INST._CURRENT (A)':    'ic',
    'PHASE_A_INST._VOLTAGE (V)':    'va',
    'PHASE_B_INST._VOLTAGE (V)':    'vb',
    'PHASE_C_INST._VOLTAGE (V)':    'vc',
    'POWER_FACTOR':                 'pf',
    'AVG._CURRENT (V)':             'avg_current',
    'AVG._VOLTAGE (V)':             'avg_voltage',
    'INST._POWER_FACTOR':           'inst_pf',
}
df = raw.rename(columns=col_map)

# --- Parse datetime (note: new files use 2-digit year, e.g. 1/31/26) ---
df['datetime'] = pd.to_datetime(
    df['date_str'].str.strip() + ' ' + df['time_str'].str.strip(),
    format='%m/%d/%y %H:%M:%S',
    errors='coerce'
)
df['serial'] = df['serial'].str.strip().str.zfill(12)

# Keep only LP rows, drop parse failures, de-duplicate cross-file overlaps
df = df[(df['obis'] == 'LP') & df['datetime'].notna()].copy()
df = df.drop_duplicates(subset=['serial', 'datetime']).copy()
df = df.sort_values(['serial', 'datetime']).reset_index(drop=True)

# Coerce numeric columns
for col in ['avg_export_kw', 'export_kwh', 'avg_import_kw', 'import_kwh']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

print(f"Shape:        {df.shape}")
print(f"Date range:   {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Unique meters:{df['serial'].nunique()}")
df[['serial', 'datetime', 'avg_import_kw', 'avg_export_kw', 'export_kwh']].head(6)

## 2. Identify Real Solar Customers
A genuine rooftop-solar meter:
- Exports power **only during daylight** (sunrise ~05:45, sunset ~18:15 in Colombo)
- Night-time export ratio < 8 %
- Exhibits a bell-shaped intra-day export profile (peak ~noon)

"Garbage" meters export at odd hours - likely data artefacts, batteries, or non-solar DG sets.

In [ ]:
# Colombo daytime: 06:00-18:00 (conservative; actual sunrise/sunset varies +/-15 min)
NIGHT_H = list(range(0, 6)) + list(range(19, 24))
DAY_H   = list(range(6, 19))

df['hour'] = df['datetime'].dt.hour

records = []
for serial, grp in df.groupby('serial'):
    exp = grp[['hour', 'avg_export_kw']].copy()
    exp['avg_export_kw'] = exp['avg_export_kw'].clip(lower=0)

    nonzero   = exp[exp['avg_export_kw'] > 0.01]
    n_nonzero = len(nonzero)

    if n_nonzero == 0:
        records.append(dict(serial=serial, has_export=False,
                            night_ratio=0, max_kw=0, n_nonzero=0,
                            day_mean_kw=0, night_mean_kw=0))
        continue

    night_exp = nonzero[nonzero['hour'].isin(NIGHT_H)]
    day_exp   = nonzero[nonzero['hour'].isin(DAY_H)]

    records.append(dict(
        serial=serial,
        has_export=True,
        night_ratio=len(night_exp) / n_nonzero,
        max_kw=grp['avg_export_kw'].max(),
        n_nonzero=n_nonzero,
        day_mean_kw=day_exp['avg_export_kw'].mean() if len(day_exp) else 0,
        night_mean_kw=night_exp['avg_export_kw'].mean() if len(night_exp) else 0,
    ))

analysis = pd.DataFrame(records).sort_values('max_kw', ascending=False)
print(analysis.to_string(index=False))

In [ ]:
# --- Apply filter thresholds ---
NIGHT_RATIO_MAX = 0.08   # <= 8% of non-zero export readings at night
MIN_EXPORT_KW   = 0.05   # must have some meaningful peak export
MIN_NONZERO     = 100    # at least 100 non-zero export intervals over the period

solar_mask = (
    analysis['has_export'] &
    (analysis['night_ratio'] <= NIGHT_RATIO_MAX) &
    (analysis['max_kw']      >= MIN_EXPORT_KW) &
    (analysis['n_nonzero']   >= MIN_NONZERO)
)

solar_serials   = analysis.loc[solar_mask, 'serial'].tolist()
garbage_serials = analysis.loc[
    analysis['has_export'] & (analysis['night_ratio'] > NIGHT_RATIO_MAX), 'serial'
].tolist()

print(f"[OK] Real solar customers : {len(solar_serials)}")
print(f"[!!] Garbage export meters: {len(garbage_serials)}")
print(f"\nSolar serials:\n{solar_serials}")

# --- Visualise hourly export fingerprint for solar vs garbage ---
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

for ax, group_serials, title, colour in [
    (axes[0], solar_serials[:6],   'Real Solar Customers (daytime bell curve)', 'steelblue'),
    (axes[1], garbage_serials[:6], 'Garbage / Night-Export Meters',             'tomato'),
]:
    for s in group_serials:
        grp = df[df['serial'] == s]
        hourly_mean = grp.groupby('hour')['avg_export_kw'].mean()
        ax.plot(hourly_mean.index, hourly_mean.values, alpha=0.7, label=s[-6:])
    ax.set_title(title)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Mean Export kW')
    ax.legend(fontsize=7, title='Serial (last 6)')
    ax.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig(DATA_DIR / 'images' / 'solar_fingerprint.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Aggregate real solar export -> substation-level 15-min target ---
solar_raw = df[df['serial'].isin(solar_serials)].copy()
solar_raw['avg_export_kw'] = solar_raw['avg_export_kw'].clip(lower=0)

agg = (
    solar_raw
    .groupby('datetime', as_index=False)
    .agg(
        total_export_kw = ('avg_export_kw', 'sum'),
        customers_reporting = ('serial', 'nunique'),
    )
)

# Build complete 15-min index for the data period to expose gaps
full_idx = pd.date_range(PERIOD_START, PERIOD_END_TS, freq='15min')
solar_ts = (
    pd.DataFrame({'datetime': full_idx})
    .merge(agg, on='datetime', how='left')
)
solar_ts['total_export_kw'] = solar_ts['total_export_kw'].fillna(0)

print(f"Time-series length : {len(solar_ts)} intervals")
print(f"Non-zero solar rows: {(solar_ts['total_export_kw'] > 0).sum()}")

# Quick overview plot - one week
sample = solar_ts[solar_ts['datetime'].between('2026-02-03', '2026-02-09')]
plt.figure(figsize=(15, 4))
plt.fill_between(sample['datetime'], sample['total_export_kw'], alpha=0.6, color='gold')
plt.plot(sample['datetime'], sample['total_export_kw'], color='darkorange', lw=1)
plt.title('Aggregated Solar Export - Sample Week (3-9 Feb 2026)')
plt.ylabel('Total Export kW')
plt.xlabel('Date / Time')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(DATA_DIR / 'images' / 'solar_aggregate_week.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Fetch 15-Min Weather Data (Open-Meteo)
**Location:** Nawala, Colombo, Sri Lanka - 6.8895 N, 79.8955 E
**API:** `historical-forecast-api.open-meteo.com` (re-analysis ERA5 + HRES blend, global 15-min)

| Variable | Description | Relevance |
|---|---|---|
| `shortwave_radiation` | GHI (W/m2) | **#1 predictor** - direct driver of PV output |
| `direct_normal_irradiance` | DNI (W/m2) | Panel-angle dependent component |
| `diffuse_radiation` | DHI (W/m2) | Scattered sky component |
| `sunshine_duration` | Seconds of sun per 15-min block | Cloud-free indicator |
| `cloud_cover` | % (0-100) | Inverse proxy for GHI |
| `temperature_2m` | C | PV efficiency drops ~0.4 %/C above 25 C |
| `precipitation` | mm | Rainy = clouds = low generation |
| `wind_speed_10m` | m/s | Panel cooling (minor effect) |
| `is_day` | 0/1 | Hard day/night gate |

In [ ]:
WEATHER_CACHE = DATA_DIR / 'weather_nawala_15min.csv'

LAT, LON = 6.8895, 79.8955

WEATHER_VARS = [
    'shortwave_radiation',
    'direct_normal_irradiance',
    'diffuse_radiation',
    'sunshine_duration',
    'cloud_cover',
    'temperature_2m',
    'precipitation',
    'wind_speed_10m',
    'is_day',
]

if WEATHER_CACHE.exists():
    print(f"Loading cached weather data from {WEATHER_CACHE.name}")
    weather_raw = pd.read_csv(WEATHER_CACHE, parse_dates=['datetime'])
else:
    print("Fetching from Open-Meteo historical forecast API ...")
    resp = requests.get(
        'https://historical-forecast-api.open-meteo.com/v1/forecast',
        params={
            'latitude':    LAT,
            'longitude':   LON,
            'start_date':  PERIOD_START,
            'end_date':    PERIOD_END,
            'minutely_15': ','.join(WEATHER_VARS),
            'timezone':    'Asia/Colombo',
            'wind_speed_unit': 'ms',
        },
        timeout=90,
    )
    resp.raise_for_status()
    data = resp.json()
    weather_raw = pd.DataFrame(data['minutely_15'])
    weather_raw.rename(columns={'time': 'datetime'}, inplace=True)
    weather_raw['datetime'] = pd.to_datetime(weather_raw['datetime'])
    weather_raw.to_csv(WEATHER_CACHE, index=False)
    print(f"Saved to {WEATHER_CACHE.name}")

# Normalise timezone info
if weather_raw['datetime'].dt.tz is not None:
    weather_raw['datetime'] = weather_raw['datetime'].dt.tz_localize(None)

print(f"Shape : {weather_raw.shape}")
print(f"Range : {weather_raw['datetime'].min()} -> {weather_raw['datetime'].max()}")
weather_raw.head()

## 4. 15-Min Resolution Filter & Per-Customer Target
Some meters report only **daily** (one reading at 00:00) rather than at 15-min resolution.
Keeping them corrupts the aggregate. We:
1. Keep only meters whose **median reading interval <= 20 min** (true load-profile data)
2. Aggregate to a **per-customer average** export (`export_per_cust = total_kW / n_reporting`),
   which is invariant to how many meters happen to report in any interval
3. Require **>= 3 meters reporting** per interval to suppress extreme outliers

In [ ]:
# --- Determine temporal resolution of each solar meter ---
solar_raw_diag = df[df['serial'].isin(solar_serials)].copy()

resolution = []
for serial, grp in solar_raw_diag.groupby('serial'):
    grp = grp.sort_values('datetime')
    diffs = grp['datetime'].diff().dropna().dt.total_seconds() / 60   # minutes
    resolution.append(dict(serial=serial, median_interval_min=diffs.median(),
                           n_readings=len(grp)))

res_df = pd.DataFrame(resolution).sort_values('median_interval_min')
print(res_df.to_string(index=False))

# Keep only genuine 15-min meters (median interval <= 20 minutes)
SOLAR_15MIN = res_df[res_df['median_interval_min'] <= 20]['serial'].tolist()
N_SOLAR = len(SOLAR_15MIN)
print(f"\n15-min solar meters : {N_SOLAR}")
print(f"Daily/other meters  : {len(solar_serials) - N_SOLAR}")

In [ ]:
# --- Re-aggregate using only true 15-min solar meters; per-customer target ---
solar_15 = df[df['serial'].isin(SOLAR_15MIN)].copy()
solar_15['avg_export_kw'] = solar_15['avg_export_kw'].clip(lower=0)

agg2 = (
    solar_15
    .groupby('datetime', as_index=False)
    .agg(
        total_export_kw = ('avg_export_kw', 'sum'),
        n_reporting     = ('serial', 'nunique'),
    )
)
agg2['export_per_cust'] = agg2['total_export_kw'] / agg2['n_reporting']

# Require at least 3 meters to report (reduces extreme outliers)
agg2_filtered = agg2[agg2['n_reporting'] >= 3].copy()
print(f"Total 15-min solar meters     : {N_SOLAR}")
print(f"Intervals with >= 3 reporters : {len(agg2_filtered):,} of {len(agg2):,}")
print(agg2_filtered['export_per_cust'].describe())

TARGET2 = 'export_per_cust'
full_idx2 = pd.date_range(PERIOD_START, PERIOD_END_TS, freq='15min')
solar_ts2 = (
    pd.DataFrame({'datetime': full_idx2})
    .merge(agg2_filtered[['datetime', TARGET2, 'n_reporting']], on='datetime', how='left')
)
print(f"\nNon-NaN target rows : {solar_ts2[TARGET2].notna().sum():,}")

# Per-customer sample-week plot
sample2 = solar_ts2[solar_ts2['datetime'].between('2026-02-03', '2026-02-09')]
plt.figure(figsize=(15, 4))
plt.fill_between(sample2['datetime'], sample2[TARGET2].fillna(0), alpha=0.5, color='gold')
plt.plot(sample2['datetime'], sample2[TARGET2].fillna(0), color='darkorange', lw=1)
plt.title('Per-Customer Average Solar Export (training target) - Sample Week Feb 2026')
plt.ylabel('Avg Export per Customer (kW)')
plt.tight_layout()
plt.savefig(DATA_DIR / 'images' / 'solar_per_customer_week.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Feature Engineering
Merge the per-customer solar target with weather, then create:
- **Astronomical features** - solar elevation angle (Spencer equation)
- **Cyclic encodings** - sin/cos of hour, day-of-year, month (avoids discontinuities)
- **Radiation lag features** - previous 15/30/60-min GHI, DNI, cloud cover
- **Rolling averages** - smoothed recent GHI / cloud cover

In [ ]:
# --- Merge solar target with weather data ---
mdf2 = solar_ts2.merge(weather_raw, on='datetime', how='inner').sort_values('datetime').reset_index(drop=True)

# --- Time features ---
mdf2['hour']     = mdf2['datetime'].dt.hour
mdf2['minute']   = mdf2['datetime'].dt.minute
mdf2['time_dec'] = mdf2['hour'] + mdf2['minute'] / 60
mdf2['doy']      = mdf2['datetime'].dt.dayofyear
mdf2['month']    = mdf2['datetime'].dt.month

# --- Solar elevation (Spencer declination, accurate to +/-0.3 deg) ---
LAT_RAD = np.radians(LAT)
B2 = 2 * np.pi * (mdf2['doy'] - 1) / 365
mdf2['declination'] = np.degrees(
    0.006918 - 0.399912*np.cos(B2) + 0.070257*np.sin(B2)
    - 0.006758*np.cos(2*B2) + 0.000907*np.sin(2*B2)
    - 0.002697*np.cos(3*B2) + 0.00148*np.sin(3*B2)
)
mdf2['hour_angle'] = (mdf2['time_dec'] - 12) * 15
mdf2['solar_elevation'] = np.degrees(np.arcsin(
    np.sin(LAT_RAD) * np.sin(np.radians(mdf2['declination'])) +
    np.cos(LAT_RAD) * np.cos(np.radians(mdf2['declination'])) *
    np.cos(np.radians(mdf2['hour_angle']))
)).clip(lower=0)

# --- Cyclic time encodings ---
mdf2['hour_sin']  = np.sin(2 * np.pi * mdf2['time_dec'] / 24)
mdf2['hour_cos']  = np.cos(2 * np.pi * mdf2['time_dec'] / 24)
mdf2['doy_sin']   = np.sin(2 * np.pi * mdf2['doy'] / 365)
mdf2['doy_cos']   = np.cos(2 * np.pi * mdf2['doy'] / 365)
mdf2['month_sin'] = np.sin(2 * np.pi * mdf2['month'] / 12)
mdf2['month_cos'] = np.cos(2 * np.pi * mdf2['month'] / 12)

# --- Radiation & cloud lag features ---
for col in ['shortwave_radiation', 'direct_normal_irradiance', 'cloud_cover']:
    mdf2[f'{col}_lag1'] = mdf2[col].shift(1)   # -15 min
    mdf2[f'{col}_lag2'] = mdf2[col].shift(2)   # -30 min
    mdf2[f'{col}_lag4'] = mdf2[col].shift(4)   # -60 min

# --- Rolling statistics ---
mdf2['ghi_roll4']   = mdf2['shortwave_radiation'].rolling(4, min_periods=1).mean()
mdf2['ghi_roll8']   = mdf2['shortwave_radiation'].rolling(8, min_periods=1).mean()
mdf2['cloud_roll4'] = mdf2['cloud_cover'].rolling(4, min_periods=1).mean()

print("Feature engineering complete - total columns:", mdf2.shape[1])
mdf2[['datetime', TARGET2, 'shortwave_radiation', 'solar_elevation', 'cloud_cover']].tail()

## 6. Model Training - LightGBM Regression

**Strategy:** daytime-only intervals (`is_day == 1`) with a **log1p target transform**.
- Night rows (export = 0) trivially dominate loss and add no signal -> excluded.
- `log1p(y)` compresses the heavy tail of peak-irradiance readings.
- **Split:** chronological 80/20 (no leakage).
- In inference: if night -> predict 0; if day -> run model -> `expm1(output)`.

In [ ]:
# Ordered feature list - order MUST be preserved for ONNX inference in ML.NET
FEATURE_COLS = [
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_radiation', 'sunshine_duration',
    'cloud_cover', 'precipitation', 'temperature_2m', 'wind_speed_10m', 'is_day',
    'solar_elevation', 'time_dec', 'doy', 'month',
    'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'month_sin', 'month_cos',
    'shortwave_radiation_lag1', 'shortwave_radiation_lag2', 'shortwave_radiation_lag4',
    'direct_normal_irradiance_lag1', 'direct_normal_irradiance_lag2',
    'cloud_cover_lag1', 'cloud_cover_lag4',
    'ghi_roll4', 'ghi_roll8', 'cloud_roll4',
]
FEATURE_COLS2 = [c for c in FEATURE_COLS if c in mdf2.columns]
print(f"Using {len(FEATURE_COLS2)} features")

# Drop NaN rows (lag features at dataset start, or NaN target = insufficient coverage)
model_df2 = mdf2[['datetime', TARGET2] + FEATURE_COLS2].dropna().copy()
model_df2 = model_df2.sort_values('datetime').reset_index(drop=True)
print(f"Total clean rows : {len(model_df2):,}")
print(f"Date range       : {model_df2['datetime'].min().date()} -> {model_df2['datetime'].max().date()}")

# --- Chronological 80/20 split ---
SPLIT_IDX  = int(len(model_df2) * 0.80)
SPLIT_DATE = model_df2.iloc[SPLIT_IDX]['datetime']
train2 = model_df2.iloc[:SPLIT_IDX]
val2   = model_df2.iloc[SPLIT_IDX:]
print(f"Train/val split date : {SPLIT_DATE.date()}")
print(f"Train: {len(train2):,} rows  ({train2['datetime'].min().date()} -> {train2['datetime'].max().date()})")
print(f"Val  : {len(val2):,} rows  ({val2['datetime'].min().date()} -> {val2['datetime'].max().date()})")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Daytime-only + log1p target ---
day_train = train2[train2['is_day'] == 1].copy()
day_val   = val2[val2['is_day'] == 1].copy()

X_tr_day = day_train[FEATURE_COLS2].values
y_tr_day = np.log1p(day_train[TARGET2].values)   # log1p transform
X_v_day  = day_val[FEATURE_COLS2].values
y_v_day  = day_val[TARGET2].values               # keep raw for evaluation

print(f"Daytime train rows : {len(X_tr_day):,}  ({day_train[TARGET2].min():.3f} - {day_train[TARGET2].max():.3f} kW/cust)")
print(f"Daytime val rows   : {len(X_v_day):,}  ({day_val[TARGET2].min():.3f} - {day_val[TARGET2].max():.3f} kW/cust)")

lgb_params_day = dict(
    objective        = 'regression_l1',   # MAE loss - robust to outliers
    metric           = 'mae',
    n_estimators     = 3000,
    learning_rate    = 0.03,
    num_leaves       = 47,
    max_depth        = 7,
    min_child_samples= 15,
    subsample        = 0.8,
    subsample_freq   = 1,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.2,
    n_jobs           = -1,
    verbose          = -1,
    random_state     = 42,
)

model2 = lgb.LGBMRegressor(**lgb_params_day)
model2.fit(
    X_tr_day, y_tr_day,
    eval_set=[(X_v_day, np.log1p(y_v_day))],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=False),
        lgb.log_evaluation(period=300),
    ],
)

# Back-transform: expm1 reverses log1p; clip negatives to zero
y_pred2_day = np.expm1(model2.predict(X_v_day)).clip(0)

rmse2_day = np.sqrt(mean_squared_error(y_v_day, y_pred2_day))
mae2_day  = mean_absolute_error(y_v_day, y_pred2_day)
r2_2_day  = r2_score(y_v_day, y_pred2_day)

print(f"\nBest iteration: {model2.best_iteration_}")
print("=" * 60)
print(f"  Daytime-only  RMSE={rmse2_day:.3f} kW/cust  MAE={mae2_day:.3f}  R2={r2_2_day:.4f}")
print(f"  (x{N_SOLAR} => fleet RMSE ~= {rmse2_day*N_SOLAR:.1f} kW)")
print("=" * 60)

# --- Feature importance + scatter ---
fi2 = (pd.DataFrame({'feature': FEATURE_COLS2, 'importance': model2.feature_importances_})
         .sort_values('importance', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].scatter(y_v_day, y_pred2_day, alpha=0.4, s=12, color='steelblue')
lim2 = max(y_v_day.max(), y_pred2_day.max()) * 1.05
axes[0].plot([0, lim2], [0, lim2], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Export per Customer (kW) - Daytime')
axes[0].set_ylabel('Predicted Export per Customer (kW)')
axes[0].set_title(f'Daytime-Only Model\nR2={r2_2_day:.3f}  RMSE={rmse2_day:.3f} kW/cust')

fi2.head(18).plot(kind='barh', x='feature', y='importance', ax=axes[1],
                   color='steelblue', legend=False)
axes[1].invert_yaxis()
axes[1].set_title('Feature Importance (Daytime Model)')
axes[1].set_xlabel('LightGBM Split Gain')

plt.tight_layout()
plt.savefig(DATA_DIR / 'images' / 'model_evaluation_v2.png', dpi=120, bbox_inches='tight')
plt.show()
print(fi2.head(10).to_string(index=False))

In [ ]:
# --- Time-series overlay for validation period ---
val_plot = day_val.copy()
val_plot['predicted_fleet_kw'] = y_pred2_day * N_SOLAR
val_plot['actual_fleet_kw']    = y_v_day * N_SOLAR

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].fill_between(val_plot['datetime'], val_plot['actual_fleet_kw'], alpha=0.4, color='gold', label=f'Actual (x{N_SOLAR} est fleet)')
axes[0].plot(val_plot['datetime'], val_plot['predicted_fleet_kw'], color='crimson', lw=1.2, label='Predicted')
axes[0].set_ylabel('Solar Export kW (fleet estimate)')
axes[0].set_title('Actual vs Predicted - Validation Period (Daytime Intervals Only)')
axes[0].legend()

weather_val = weather_raw[weather_raw['datetime'].isin(val_plot['datetime'])].copy()
weather_val = weather_val.merge(val_plot[['datetime']], on='datetime')
ghi_max = weather_val['shortwave_radiation'].max()
axes[1].fill_between(weather_val['datetime'], weather_val['shortwave_radiation'] / (ghi_max + 1e-3),
                     alpha=0.5, color='darkorange', label='GHI (norm.)')
axes[1].plot(weather_val['datetime'], weather_val['cloud_cover'] / 100,
             color='steelblue', lw=0.9, label='Cloud Cover fraction')
axes[1].set_ylabel('Normalised')
axes[1].set_xlabel('Date / Time')
axes[1].legend()

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(DATA_DIR / 'images' / 'solar_forecast_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. ONNX Export for ML.NET

The trained LightGBM model is converted to **ONNX opset 12** via `onnxmltools`, preserving the
`float_input` -> `variable` tensor contract used by the original model.

```csharp
var pipeline = mlContext.Transforms.ApplyOnnxModel(
    modelFile:         "solar_forecast_model.onnx",
    outputColumnNames: new[] { "variable" },
    inputColumnNames:  new[] { "float_input" });
```
The input must be a `float[]` of length 29 in the exact order listed in `model_metadata.json`.

In [ ]:
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as rt

ONNX_PATH  = DATA_DIR / 'solar_forecast_model.onnx'
N_FEATURES = len(FEATURE_COLS2)

# Convert the LightGBM booster directly (opset 12). The single output is renamed
# to the standard 'variable' tensor name expected by ML.NET ApplyOnnxModel.
onnx_model = onnxmltools.convert_lightgbm(
    model2.booster_,
    initial_types=[('float_input', FloatTensorType([None, N_FEATURES]))],
    target_opset=12,
)
onnx_model.graph.output[0].name = 'variable'
with open(ONNX_PATH, 'wb') as f:
    f.write(onnx_model.SerializeToString())
print(f"Saved ONNX -> {ONNX_PATH}")

# --- Verify with onnxruntime ---
sess = rt.InferenceSession(str(ONNX_PATH))
sample_in   = X_v_day[:5].astype(np.float32)
input_name  = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name
raw_out  = sess.run([output_name], {input_name: sample_in})[0].flatten()
onnx_pred = np.expm1(raw_out).clip(0)
lgb_pred  = np.expm1(model2.predict(X_v_day[:5])).clip(0)

print(f"\nONNX verified [OK]")
print(f"  Input tensor  : '{input_name}'  shape {sess.get_inputs()[0].shape}")
print(f"  Output tensor : '{output_name}' shape {sess.get_outputs()[0].shape}")
print(f"  Max abs diff (ONNX vs LightGBM) : {np.abs(onnx_pred - lgb_pred).max():.6f} kW/cust")

## 8. Save Model Metadata JSON
Emits the feature-order contract, ONNX tensor names, post-processing rules, user-scaling notes, and validation metrics consumed by the ML.NET backend.

In [ ]:
# Estimate fleet average installed capacity from peak observed per-customer export.
peak_export   = float(day_train[TARGET2].max())
FLEET_AVG_KWP = round(peak_export / 0.82, 1)
train_period  = f"{day_train['datetime'].min().date()} to {day_train['datetime'].max().date()}"
val_period    = f"{day_val['datetime'].min().date()} to {day_val['datetime'].max().date()}"
print(f"Peak training export per customer : {peak_export:.3f} kW")
print(f"Estimated fleet avg capacity      : {FLEET_AVG_KWP} kWp")

meta = {
    "project": "SunShift",
    "description": "Solar generation forecasting - LECO Nawala substation, Colombo, Sri Lanka",
    "onnx_path": "solar_forecast_model.onnx",
    "onnx_opset": 12,
    "input_tensor_name": "float_input",
    "input_shape": [None, len(FEATURE_COLS2)],
    "output_tensor_name": output_name,
    "output_shape": [None, 1],
    "output_transform": "log1p",
    "output_decode": "base_kw_per_cust = max(0, exp(raw_output) - 1)",
    "output_unit": "kW per customer assuming fleet-average 5 kWp horizontal-reference system",
    "n_solar_customers": N_SOLAR,
    "user_scaling": {
        "fleet_avg_capacity_kwp": FLEET_AVG_KWP,
        "fleet_avg_capacity_note": (
            f"Estimated from peak observed per-customer export ({peak_export:.2f} kW) "
            f"divided by assumed 0.82 peak efficiency. Replace with real fleet average "
            f"if LECO meter registration data (kWp per installation) becomes available."),
        "poa_model": "isotropic sky model (Hay-Davies diffuse) with albedo=0.20",
        "poa_formula": {
            "cos_aoi": "sin(elev)*cos(tilt) + cos(elev)*sin(tilt)*cos(solar_azimuth - panel_azimuth)",
            "poa_direct": "DNI * max(0, cos_aoi)",
            "poa_diffuse": "DHI * (1 + cos(tilt)) / 2",
            "poa_reflected": "GHI * 0.20 * (1 - cos(tilt)) / 2",
            "poa_total": "poa_direct + poa_diffuse + poa_reflected"},
        "solar_azimuth_note": (
            "Solar azimuth must be computed alongside solar_elevation using the same "
            "Spencer equation. Convention: 0=N, 90=E, 180=S, 270=W (meteorological). "
            "formula: atan2(sin(H), cos(H)*sin(lat) - tan(decl)*cos(lat)) + 180 deg, mod 360"),
        "panel_azimuth_note": (
            "Azimuth property from user profile: accepts compass string (N/NE/E/SE/S/SW/W/NW) "
            "or numeric degrees (0=N, 90=E, 180=S, 270=W). For Sri Lanka (6.9 N), "
            "south-facing (180 deg) is optimal."),
        "inverter_clamp": "Math.Min(predicted_kw, user.InverterCapacityKw)"},
    "feature_cols": FEATURE_COLS2,
    "n_features": len(FEATURE_COLS2),
    "feature_notes": {
        "solar_elevation": "Spencer-equation sun elevation in degrees; lat=6.8895, lon=79.8955",
        "declination": "Solar declination angle (Spencer) in degrees",
        "hour_angle": "Hour angle in degrees from solar noon",
        "hour_sin / hour_cos": "sin/cos of 2*pi*(hour+minute/60)/24",
        "doy_sin / doy_cos": "sin/cos of 2*pi*dayofyear/365",
        "month_sin / month_cos": "sin/cos of 2*pi*month/12",
        "time_dec": "Hour decimal (hour + minute/60)",
        "doy": "Day of year (1-365)", "month": "Month (1-12)",
        "lag features": "Shift by 1/2/4 x 15-min intervals of GHI (shortwave_radiation) / DNI / cloud_cover",
        "rolling means": "ghi_roll4 = mean(GHI, 4 periods), ghi_roll8 = mean(GHI, 8 periods), cloud_roll4 = mean(cloud_cover, 4 periods)",
        "is_day": "1 if sun is above horizon, else 0 (from Open-Meteo)"},
    "training_data_source": "LECO smart meter load profile - Nawala substation (files: LP-AZ0247-2026-01.csv, LP-AZ0247-2026-02.csv)",
    "weather_source": "Open-Meteo Historical Forecast API (minutely_15)",
    "weather_lat": 6.8895, "weather_lon": 79.8955, "weather_timezone": "Asia/Colombo",
    "weather_variables": WEATHER_VARS,
    "algorithm": "LightGBM LGBMRegressor (daytime-only, log1p target)",
    "target_variable": "export_per_cust = total_solar_export_kw / n_reporting_meters",
    "training_period": train_period, "validation_period": val_period,
    "train_rows": int(len(X_tr_day)), "val_rows": int(len(X_v_day)),
    "training_intervals_used": "daytime only (is_day==1)",
    "best_iteration": int(model2.best_iteration_),
    "val_rmse_kw_per_customer": round(float(rmse2_day), 4),
    "val_mae_kw_per_customer": round(float(mae2_day), 4),
    "val_r2": round(float(r2_2_day), 4),
    "val_fleet_rmse_kw": round(float(rmse2_day * N_SOLAR), 2),
    "solar_meter_serials": [str(s) for s in SOLAR_15MIN],
    "solar_identification_method": "night_ratio <= 0.08 AND max_kw >= 0.05 AND n_nonzero >= 100; then median reading interval <= 20 min; then >= 3 meters reporting per interval",
    "data_quality_notes": (
        "Retrained on clean Jan + Feb 2026 LECO export (the prior dataset had corrupted records). "
        "Meters with constant non-zero export at all hours are excluded as cumulative-register artefacts. "
        "Daily-resolution meters are dropped by the 15-min median-interval filter. Per-customer target "
        "is invariant to meter communication gaps. Performance expected to improve further with 6+ months "
        "of consistent smart-meter data."),
    "mlnet_integration": {
        "pipeline": "mlContext.Transforms.ApplyOnnxModel(modelFile: 'solar_forecast_model.onnx', outputColumnName: 'variable', inputColumnName: 'float_input')",
        "base_post_process": "double baseKw = Math.Max(0.0, Math.Exp(rawOutput[0]) - 1.0);",
        "user_scaling": "See SolarForecastScaler.ScaleToUserSystem() in the mobile-app backend.",
        "note": f"Input must be float32[] of length {len(FEATURE_COLS2)}, features ordered exactly as feature_cols list above."},
}

META_PATH = DATA_DIR / 'model_metadata.json'
with open(META_PATH, 'w') as f:
    json.dump(meta, f, indent=2)

print(f"Metadata saved -> {META_PATH}")
print(f"\nModel summary:")
print(f"  ONNX file          : {ONNX_PATH.name}  ({ONNX_PATH.stat().st_size / 1024:.0f} KB)")
print(f"  Features           : {len(FEATURE_COLS2)}")
print(f"  Best iter          : {model2.best_iteration_}")
print(f"  Val RMSE           : {rmse2_day:.3f} kW/cust  -> fleet {rmse2_day*N_SOLAR:.1f} kW")
print(f"  Val MAE            : {mae2_day:.3f} kW/cust")
print(f"  Val R2             : {r2_2_day:.4f}")
print(f"  Fleet avg capacity : {FLEET_AVG_KWP} kWp (estimated)")
print(f"\nFeature list in order (must match ML.NET input):")
for i, f in enumerate(FEATURE_COLS2):
    print(f"  [{i:02d}] {f}")